# Bilbao Bar Crawl Optimizer
## Experimento reproducible de rutas peatonales

Este cuaderno lee los artefactos generados por `scripts/` y resume el escenario, las decisiones de modelado y la comparación de algoritmos. No sustituye al pipeline: sirve para inspeccionar y comunicar sus resultados.

## 1. Escenario y decisiones

- Zona: Siete Calles, Plaza Nueva, El Arenal y calles intermedias.
- Recorrido: abierto, salida fija desde Plaza Nueva y final libre.
- Coste: distancia sobre red peatonal dirigida.
- Acceso: un nodo virtual independiente por establecimiento, proyectado sobre la arista peatonal más cercana.
- Simulación: 4,5 km/h caminando y 15 minutos por establecimiento.

La solución mostrada es la mejor encontrada por cada método; no se afirma optimalidad global sin una garantía matemática.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
processed = DATA / 'processed'

def load_json(name):
    return json.loads((processed / name).read_text(encoding='utf-8'))

network = load_json('pedestrian-network-report.json')
matrix = load_json('distance-matrix-report.json')
simulations = load_json('optimization-simulations.json')

print(f"Establecimientos: {network['venue_count']}")
print(f"Nodos de red: {network['nodes']:,} · aristas: {network['edges']:,}")
print(f"Nodos virtuales de acceso: {network['virtual_access_nodes']}")
print(f"Pares alcanzables: {matrix['reachable_ordered_pairs']:,} / {matrix['ordered_pairs']:,}")

In [ ]:
print(f"{'Algoritmo':35} {'km':>8} {'caminar':>12} {'total':>12}")
print('-' * 72)
for item in sorted(simulations['simulations'], key=lambda row: row['total_duration_min']):
    print(f"{item['algorithm']:35} {item['distance_m'] / 1000:8.2f} {item['walking_time_min']:9.2f} min {item['total_duration_min']:9.2f} min")

## 2. Lectura de resultados

La distancia caminando y el tiempo total responden a preguntas distintas. El tiempo total suma una parada fija de 15 minutos por establecimiento; por eso la parte dominante de la simulación no cambia entre algoritmos. La animación representa únicamente el recorrido peatonal y no simula el tiempo detenido.

In [ ]:
best = min(simulations['simulations'], key=lambda row: row['total_duration_min'])
random = next(row for row in simulations['simulations'] if row['algorithm'] == 'random')
improvement = (random['distance_m'] - best['distance_m']) / random['distance_m'] * 100
print(f"Mejor solución encontrada: {best['algorithm']}")
print(f"Mejora frente a la referencia aleatoria: {improvement:.1f}%")
print(f"Tiempo total estimado: {best['total_duration_min']:.2f} minutos")